# Demand Forecasting — Analysis

Notebook này chỉ dùng để kiểm tra dữ liệu, EDA và xem kết quả. Logic sinh dữ liệu, PostgreSQL, feature engineering, train và diagnostics nằm trong `src/`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from demand_forecasting.config import get_settings
from demand_forecasting.database import database_row_count, load_raw_demand

settings = get_settings()
settings

## 1. Kiểm tra PostgreSQL

In [ ]:
rows = database_row_count(settings)
print(f"PostgreSQL rows: {rows:,}")

## 2. Đọc mẫu dữ liệu raw

In [ ]:
raw = load_raw_demand(settings, limit=100_000)
display(raw.head())
display(raw.dtypes)

In [ ]:
summary = raw.groupby("travel_mode", observed=True)["total_demand"].agg(
    ["count", "mean", "std", "min", "median", "max"]
)
display(summary)

## 3. Đọc kết quả train

In [ ]:
summary_path = settings.output_dir / "weekly_retrain_summary.csv"
if summary_path.exists():
    training_summary = pd.read_csv(summary_path)
    display(training_summary)
else:
    print("Chưa có weekly_retrain_summary.csv. Đợi bootstrap hoặc Airflow weekly retrain.")

## 4. Stability theo ngày

In [ ]:
daily_path = settings.output_dir / "diagnostics" / "daily_stability.csv"
if daily_path.exists():
    daily = pd.read_csv(daily_path, parse_dates=["date"])
    for keys, group in daily.groupby(["horizon", "travel_mode", "model"]):
        group.plot(x="date", y="WMAPE_%", figsize=(10, 3), title=" | ".join(keys))
        plt.grid(alpha=0.25)
        plt.show()
else:
    print("Chưa có diagnostics. Hãy chạy scripts/evaluate_models.py.")